In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [9]:
df = pd.read_csv('../data/bank_churn_segmented.csv')

feature_cols = [c for c in df.columns
               if c not in ['Exited', 'RowNumber', 'CustomerId', 'Surname']]
X = df[feature_cols]
Y = df['Exited']

print(f'Features: {X.shape[1]} columns, {X.shape[0]} rows')
print(f'Target distribution: {Y.value_counts().to_dict()}')

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size = 0.2, random_state = 42, stratify = Y
)

print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Features: 17 columns, 10000 rows
Target distribution: {0: 7963, 1: 2037}
Training samples: 8000
Test samples: 2000


In [13]:
#applying smote only on the training data to handle class imbalance

from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state = 42)
X_train_res, Y_train_res = smote.fit_resample(X_train_scaled, Y_train)
print('Before SMOTE:', Y_train.value_counts().to_dict())
print('After SMOTE:', pd.Series(Y_train_res).value_counts().to_dict())

Before SMOTE: {0: 6370, 1: 1630}
After SMOTE: {1: 6370, 0: 6370}


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

lr_model = LogisticRegression(max_iter = 1000, random_state = 42)
lr_model.fit(X_train_res, Y_train_res)

y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:,1]

print('--- Logistic Regression Results ---')
print(classification_report(Y_test, y_pred_lr, target_names = ['Stayed', 'Churned']))
print(f'ROC-AUC Score: {roc_auc_score(Y_test, y_prob_lr):.4f}')

--- Logistic Regression Results ---
              precision    recall  f1-score   support

      Stayed       0.90      0.75      0.82      1593
     Churned       0.40      0.67      0.51       407

    accuracy                           0.73      2000
   macro avg       0.65      0.71      0.66      2000
weighted avg       0.80      0.73      0.75      2000

ROC-AUC Score: 0.7819
